# MV-Swin-T: Mammogram Multi-Task Classification
### Optimized Version — 3 perbaikan untuk imbalanced dataset:
1. ✅ **Focal Loss** — ganti CrossEntropy, paksa model fokus ke sampel sulit (kelas minoritas)
2. ✅ **Macro F1 sebagai checkpoint** — ganti val_loss, model terbaik = F1 tertinggi semua kelas
3. ✅ **Dynamic alpha/beta** — bobot loss BI-RADS vs Density menyesuaikan performa tiap task
4. ✅ Semua perbaikan versi sebelumnya tetap dipertahankan

In [ ]:
import shutil, os

shutil.copy(
    '/kaggle/input/datasets/vickyoktafrian/model-mv-swin-t/dataset_classification_vindr.py',
    '/kaggle/working/dataset_classification_vindr.py'
)

if os.path.exists('/kaggle/working/models'):
    shutil.rmtree('/kaggle/working/models')

shutil.copytree(
    '/kaggle/input/datasets/vickyoktafrian/model-mv-swin-t/models',
    '/kaggle/working/models'
)

print('✅ File code berhasil di-copy ke /kaggle/working/')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision.transforms import v2 as transforms
from torch.utils.data import DataLoader
import time
import numpy as np
from sklearn import metrics
from sklearn.preprocessing import label_binarize

In [ ]:
from dataset_classification_vindr import MakeDataset_VinDr_classification
from models.mvswintransformer import MVSwinTransformer

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('Available GPUs:', torch.cuda.device_count())

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────
extension   = '.png'
target_size = 224
window_size = 7

batch_size    = 16
learning_rate = 1e-4
epochs        = 50
weight_decay  = 1e-3

# [SOLUSI 3] Bobot awal alpha/beta — akan disesuaikan dinamis tiap epoch
alpha = 1.0   # bobot loss BI-RADS
beta  = 1.0   # bobot loss Density
DYNAMIC_WEIGHT_LR = 0.1   # seberapa cepat alpha/beta berubah (0.0 = mati, 1.0 = agresif)

# [SOLUSI 1] Focal Loss gamma — semakin besar, semakin fokus ke sampel sulit
# Rekomendasi: 2.0 untuk imbalance sedang, coba 3.0 jika Density A masih buruk
FOCAL_GAMMA = 2.0

# Early stopping: berhenti jika val Macro F1 tidak membaik selama N epoch
EARLY_STOP_PATIENCE = 10

In [ ]:
import os

if os.path.exists('/kaggle/input'):
    image_dir     = '/kaggle/input/datasets/vickyoktafrian/dataset-preprocessed/dataset_preprocessed/dataset_preprocessed'
    label_dir_csv = '/kaggle/input/datasets/vickyoktafrian/dataset-preprocessed/breast-level_annotations.csv'
else:
    image_dir     = './dataset_preprocessed'
    label_dir_csv = './breast-level_annotations.csv'

print('Image dir :', image_dir)
print('Label CSV :', label_dir_csv)
print('Image dir exists:', os.path.exists(image_dir))
print('CSV exists:', os.path.exists(label_dir_csv))

In [ ]:
# ── DATA LOADERS ───────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
    transforms.ColorJitter(brightness=0.2),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

transform_val_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_dataset = MakeDataset_VinDr_classification(
    image_dir=image_dir, label_dir_csv=label_dir_csv,
    transform=transform_train, mode='train', target_size=target_size
)

from torch.utils.data import random_split
val_ratio  = 0.2
val_size   = int(len(train_dataset) * val_ratio)
train_size = len(train_dataset) - val_size
train_subset, val_subset = random_split(
    train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
val_subset.dataset.transform = transform_val_test

test_dataset = MakeDataset_VinDr_classification(
    image_dir=image_dir, label_dir_csv=label_dir_csv,
    transform=transform_val_test, mode='test', target_size=target_size
)

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_subset,   batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f'Train : {train_size} | Val: {val_size} | Test: {len(test_dataset)}')

In [ ]:
# ── MODEL ──────────────────────────────────────────────────
model = MVSwinTransformer(img_size=224, window_size=7).to(device)

pytorch_total_params     = sum(p.numel() for p in model.parameters())
pytorch_total_trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {pytorch_total_params  // 10**6} M')
print(f'Trainable params : {pytorch_total_trainable // 10**6} M')

In [ ]:
# ── PRETRAINED SWIN-T WEIGHTS ──────────────────────────────
import glob

pth_candidates = glob.glob('/kaggle/input/**/swin_tiny_patch4_window7_224.pth', recursive=True)
if not pth_candidates:
    pth_candidates = glob.glob('/kaggle/input/**/*.pth', recursive=True)

print('File .pth ditemukan:', pth_candidates)
assert pth_candidates, '❌ File .pth tidak ditemukan!'
pretrained_pth = pth_candidates[0]
print(f'Memuat pretrained Swin-T weights dari: {pretrained_pth}')

pretrained_sd = torch.load(pretrained_pth, map_location='cpu')
if isinstance(pretrained_sd, dict) and 'model' in pretrained_sd:
    pretrained_sd = pretrained_sd['model']
elif isinstance(pretrained_sd, dict) and 'state_dict' in pretrained_sd:
    pretrained_sd = pretrained_sd['state_dict']

model_sd = model.state_dict()
loaded, skipped = 0, 0
new_sd = {}

for key, val in pretrained_sd.items():
    if key.startswith('patch_embed.'):
        suffix = key[len('patch_embed.'):]
        for view in ['patch_embed_1', 'patch_embed_2']:
            target = f'{view}.{suffix}'
            if target in model_sd and model_sd[target].shape == val.shape:
                new_sd[target] = val
                loaded += 1
    elif key.startswith('layers.2.') or key.startswith('layers.3.'):
        suffix    = key[len('layers.'):]
        fused_idx = int(suffix.split('.')[0]) - 2
        rest      = '.'.join(suffix.split('.')[1:])
        target    = f'layers_fused.{fused_idx}.{rest}'
        if target in model_sd and model_sd[target].shape == val.shape:
            new_sd[target] = val
            loaded += 1
        else:
            skipped += 1
    else:
        skipped += 1

model_sd.update(new_sd)
model.load_state_dict(model_sd)
print(f'✅ Pretrained weights injected: {loaded} keys loaded, {skipped} skipped')
del pretrained_sd

In [ ]:
# ── SOLUSI 1: FOCAL LOSS ───────────────────────────────────
#
# Focal Loss = -alpha_t * (1 - p_t)^gamma * log(p_t)
#
# Cara kerjanya:
#   - p_t         : probabilitas model pada kelas yang benar
#   - (1-p_t)^γ   : faktor modulator
#       → jika model sudah yakin (p_t ≈ 0.95): (1-0.95)^2 = 0.0025 → loss dikecilkan drastis
#       → jika model tidak yakin (p_t ≈ 0.2) : (1-0.20)^2 = 0.64   → loss hampir tidak berubah
#   - class_weights tetap dipakai (alpha_t) untuk mengatasi imbalance frekuensi
#
# Kombinasi class_weights + gamma = dua lapisan perlindungan untuk kelas minoritas

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, reduction='mean'):
        super().__init__()
        self.gamma     = gamma
        self.weight    = weight      # class weights (alpha_t)
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Step 1: Hitung log-softmax untuk stabilitas numerik
        log_prob = F.log_softmax(inputs, dim=1)                      # (B, C)

        # Step 2: Ambil log-prob untuk kelas yang benar
        log_prob_true = log_prob.gather(1, targets.unsqueeze(1)).squeeze(1)  # (B,)

        # Step 3: Hitung p_t dan faktor modulator (1-p_t)^gamma
        prob_true     = log_prob_true.exp()                           # p_t
        focal_factor  = (1.0 - prob_true) ** self.gamma              # (1-p_t)^γ

        # Step 4: Focal loss sebelum class weighting
        focal_loss    = -focal_factor * log_prob_true                # (B,)

        # Step 5: Terapkan class weights jika ada
        if self.weight is not None:
            w          = self.weight[targets]                        # ambil weight per sampel
            focal_loss = focal_loss * w

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

print(f'✅ FocalLoss class siap (gamma={FOCAL_GAMMA})')

In [ ]:
# ── CLASS WEIGHTS (tetap dipakai, dikombinasi dengan Focal Loss) ──
# Menggunakan sqrt-inverse weighting yang sudah ada sebelumnya

def compute_class_weights_balanced(dataset, num_classes, label_key, max_weight=10.0):
    counts  = torch.zeros(num_classes)
    pairs   = dataset.dataset.pairs if hasattr(dataset, 'dataset') else dataset.pairs
    indices = dataset.indices if hasattr(dataset, 'indices') else range(len(pairs))
    for i in indices:
        counts[pairs[i][label_key]] += 1
    weights = 1.0 / (counts.sqrt() + 1e-6)
    weights = weights / weights.mean()
    weights = weights.clamp(max=max_weight)
    return weights

weights_birads  = compute_class_weights_balanced(train_subset, 5, 'label_birads').to(device)
weights_density = compute_class_weights_balanced(train_subset, 4, 'label_density').to(device)

print('Class weights BI-RADS :', weights_birads.cpu().numpy().round(3))
print('Class weights Density :', weights_density.cpu().numpy().round(3))

In [ ]:
# ── OPTIMIZER, SCHEDULER, LOSS (Focal Loss) ────────────────
optimizer         = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler         = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=epochs, eta_min=1e-6
)

# [SOLUSI 1] Ganti CrossEntropyLoss → FocalLoss
# class_weights tetap dipakai sebagai alpha_t dalam Focal Loss
criterion_birads  = FocalLoss(gamma=FOCAL_GAMMA, weight=weights_birads)
criterion_density = FocalLoss(gamma=FOCAL_GAMMA, weight=weights_density)

print('Optimizer  : AdamW')
print('Scheduler  : CosineAnnealingLR')
print(f'Loss       : FocalLoss (gamma={FOCAL_GAMMA}) + class weights')

In [ ]:
# ── SOLUSI 2 + 3: TRAINING LOOP dengan Macro F1 Checkpoint
#                 dan Dynamic Alpha/Beta ──────────────────────
#
# Perubahan dari versi sebelumnya:
#   [SOLUSI 2] best_val_loss  → best_macro_f1  (simpan model terbaik berdasarkan Macro F1)
#              early stopping → trigger jika Macro F1 tidak naik (bukan val_loss turun)
#   [SOLUSI 3] alpha/beta tetap, tapi tiap epoch disesuaikan:
#              task yang F1-nya lebih rendah mendapat bobot lebih besar
#              rumus: alpha_new = alpha + lr_dw * (f1_density - f1_birads)
#                     beta_new  = beta  + lr_dw * (f1_birads  - f1_density)
#              lalu dinormalisasi agar alpha + beta = 2.0

model_save_path    = '/kaggle/working/best_model.pth' if os.path.exists('/kaggle/input') else './best_model.pth'
print('Model akan disimpan di:', model_save_path)

best_macro_f1      = 0.0     # [SOLUSI 2] ganti dari best_val_loss = inf
early_stop_counter = 0
history            = []

# [SOLUSI 3] Inisialisasi alpha/beta yang akan diupdate dinamis
alpha_dynamic = alpha
beta_dynamic  = beta

for epoch in range(1, epochs + 1):
    since = time.time()
    print('-' * 60)
    print(f'Epoch [{epoch}/{epochs}] | alpha={alpha_dynamic:.3f} | beta={beta_dynamic:.3f}')

    # ── TRAIN ────────────────────────────────────────────────
    model.train()
    train_loss_sum  = 0.0
    correct_birads  = 0
    correct_density = 0
    total           = 0

    for inputs_cc, inputs_mlo, labels_birads, labels_density in train_loader:
        inputs_cc      = inputs_cc.float().to(device)
        inputs_mlo     = inputs_mlo.float().to(device)
        labels_birads  = labels_birads.long().to(device)
        labels_density = labels_density.long().to(device)

        optimizer.zero_grad()
        pred_birads, pred_density = model(inputs_cc, inputs_mlo)

        loss_birads  = criterion_birads(pred_birads,   labels_birads)
        loss_density = criterion_density(pred_density, labels_density)

        # [SOLUSI 3] Pakai alpha/beta yang sudah disesuaikan secara dinamis
        total_loss = alpha_dynamic * loss_birads + beta_dynamic * loss_density

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss_sum  += total_loss.item()
        total           += labels_birads.size(0)
        correct_birads  += (pred_birads.argmax(dim=1)  == labels_birads).sum().item()
        correct_density += (pred_density.argmax(dim=1) == labels_density).sum().item()

    train_loss        = train_loss_sum / len(train_loader)
    train_acc_birads  = 100 * correct_birads  / total
    train_acc_density = 100 * correct_density / total

    print(f'  [TRAIN] Loss: {train_loss:.4f} | '
          f'Acc BI-RADS: {train_acc_birads:.2f}% | '
          f'Acc Density: {train_acc_density:.2f}%')

    # ── VALIDATION ───────────────────────────────────────────
    model.eval()
    val_loss_sum    = 0.0
    correct_birads  = 0
    correct_density = 0
    val_total       = 0

    # [SOLUSI 2] Kumpulkan semua prediksi untuk hitung Macro F1
    all_pred_b = []
    all_true_b = []
    all_pred_d = []
    all_true_d = []

    with torch.no_grad():
        for inputs_cc, inputs_mlo, labels_birads, labels_density in val_loader:
            inputs_cc      = inputs_cc.float().to(device)
            inputs_mlo     = inputs_mlo.float().to(device)
            labels_birads  = labels_birads.long().to(device)
            labels_density = labels_density.long().to(device)

            pred_birads, pred_density = model(inputs_cc, inputs_mlo)

            loss_birads  = criterion_birads(pred_birads,   labels_birads)
            loss_density = criterion_density(pred_density, labels_density)
            val_loss_sum += (alpha_dynamic * loss_birads + beta_dynamic * loss_density).item()

            val_total       += labels_birads.size(0)
            correct_birads  += (pred_birads.argmax(dim=1)  == labels_birads).sum().item()
            correct_density += (pred_density.argmax(dim=1) == labels_density).sum().item()

            all_pred_b.extend(pred_birads.argmax(dim=1).cpu().numpy())
            all_true_b.extend(labels_birads.cpu().numpy())
            all_pred_d.extend(pred_density.argmax(dim=1).cpu().numpy())
            all_true_d.extend(labels_density.cpu().numpy())

    val_loss        = val_loss_sum / len(val_loader)
    val_acc_birads  = 100 * correct_birads  / val_total
    val_acc_density = 100 * correct_density / val_total

    # [SOLUSI 2] Hitung Macro F1 per task
    f1_birads  = metrics.f1_score(all_true_b, all_pred_b, average='macro', zero_division=0)
    f1_density = metrics.f1_score(all_true_d, all_pred_d, average='macro', zero_division=0)
    # Gabungan: rata-rata Macro F1 kedua task sebagai single checkpoint metric
    combined_f1 = (f1_birads + f1_density) / 2.0

    time_elapsed = time.time() - since
    curr_lr      = optimizer.param_groups[0]['lr']

    print(f'  [VAL]   Loss: {val_loss:.4f} | '
          f'Acc BI-RADS: {val_acc_birads:.2f}% | '
          f'Acc Density: {val_acc_density:.2f}% | '
          f'LR: {curr_lr:.6f} | '
          f'Time: {time_elapsed//60:.0f}m {time_elapsed%60:.0f}s')
    print(f'  [F1]    BI-RADS Macro F1: {f1_birads:.4f} | '
          f'Density Macro F1: {f1_density:.4f} | '
          f'Combined: {combined_f1:.4f}')

    history.append({
        'epoch'             : epoch,
        'train_loss'        : train_loss,
        'val_loss'          : val_loss,
        'train_acc_birads'  : train_acc_birads,
        'val_acc_birads'    : val_acc_birads,
        'train_acc_density' : train_acc_density,
        'val_acc_density'   : val_acc_density,
        'f1_birads'         : f1_birads,
        'f1_density'        : f1_density,
        'combined_f1'       : combined_f1,
        'alpha'             : alpha_dynamic,
        'beta'              : beta_dynamic,
    })

    scheduler.step()

    # [SOLUSI 3] Update alpha/beta berdasarkan selisih F1 kedua task
    # Logika: task yang F1-nya lebih rendah → naikkan bobotnya
    f1_gap         = f1_birads - f1_density          # positif = BI-RADS lebih bagus
    alpha_dynamic  = alpha_dynamic - DYNAMIC_WEIGHT_LR * f1_gap   # turunkan alpha jika BI-RADS unggul
    beta_dynamic   = beta_dynamic  + DYNAMIC_WEIGHT_LR * f1_gap   # naikkan beta  jika BI-RADS unggul
    # Normalisasi agar alpha + beta = 2.0 dan keduanya >= 0.3 (batas bawah)
    alpha_dynamic  = max(0.3, min(3.0, alpha_dynamic))
    beta_dynamic   = max(0.3, min(3.0, beta_dynamic))

    # [SOLUSI 2] Simpan model terbaik berdasarkan Combined Macro F1 (bukan val_loss)
    if combined_f1 > best_macro_f1:
        best_macro_f1      = combined_f1
        early_stop_counter = 0
        torch.save(model.state_dict(), model_save_path)
        print(f'  ✅ Model terbaik disimpan (Combined Macro F1: {combined_f1:.4f})')
    else:
        early_stop_counter += 1
        print(f'  ⏳ Early stop counter: {early_stop_counter}/{EARLY_STOP_PATIENCE} '
              f'(best F1: {best_macro_f1:.4f})')
        if early_stop_counter >= EARLY_STOP_PATIENCE:
            print(f'  🛑 Early stopping triggered at epoch {epoch}')
            break

print('\n✅ Training selesai!')
print(f'Best Combined Macro F1: {best_macro_f1:.4f}')

In [ ]:
# ── PLOT TRAINING HISTORY (termasuk F1 dan alpha/beta) ─────
import matplotlib.pyplot as plt

epochs_list      = [h['epoch']              for h in history]
train_losses     = [h['train_loss']         for h in history]
val_losses       = [h['val_loss']           for h in history]
train_acc_b      = [h['train_acc_birads']   for h in history]
val_acc_b        = [h['val_acc_birads']     for h in history]
train_acc_d      = [h['train_acc_density']  for h in history]
val_acc_d        = [h['val_acc_density']    for h in history]
f1_b_list        = [h['f1_birads']          for h in history]
f1_d_list        = [h['f1_density']         for h in history]
combined_f1_list = [h['combined_f1']        for h in history]
alpha_list       = [h['alpha']              for h in history]
beta_list        = [h['beta']               for h in history]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0,0].plot(epochs_list, train_losses, label='Train Loss')
axes[0,0].plot(epochs_list, val_losses,   label='Val Loss')
axes[0,0].set_title('Loss'); axes[0,0].legend(); axes[0,0].set_xlabel('Epoch')

axes[0,1].plot(epochs_list, train_acc_b, label='Train')
axes[0,1].plot(epochs_list, val_acc_b,   label='Val')
axes[0,1].set_title('Accuracy BI-RADS'); axes[0,1].legend(); axes[0,1].set_xlabel('Epoch')

axes[0,2].plot(epochs_list, train_acc_d, label='Train')
axes[0,2].plot(epochs_list, val_acc_d,   label='Val')
axes[0,2].set_title('Accuracy Density'); axes[0,2].legend(); axes[0,2].set_xlabel('Epoch')

# [BARU] Plot Macro F1 tiap task
axes[1,0].plot(epochs_list, f1_b_list,        label='BI-RADS Macro F1')
axes[1,0].plot(epochs_list, f1_d_list,        label='Density Macro F1')
axes[1,0].plot(epochs_list, combined_f1_list, label='Combined', linestyle='--')
axes[1,0].set_title('Macro F1 (checkpoint metric)'); axes[1,0].legend(); axes[1,0].set_xlabel('Epoch')

# [BARU] Plot perubahan alpha/beta
axes[1,1].plot(epochs_list, alpha_list, label='alpha (BI-RADS)')
axes[1,1].plot(epochs_list, beta_list,  label='beta (Density)')
axes[1,1].set_title('Dynamic alpha/beta'); axes[1,1].legend(); axes[1,1].set_xlabel('Epoch')
axes[1,1].set_ylim(0, 3.5)

# [BARU] Gap F1 antar task
gap = [abs(b - d) for b, d in zip(f1_b_list, f1_d_list)]
axes[1,2].plot(epochs_list, gap, color='red', label='|F1_birads - F1_density|')
axes[1,2].set_title('F1 gap antar task (ideal: mendekati 0)'); axes[1,2].legend(); axes[1,2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('training_history_optimized.png', dpi=150)
plt.show()
print('Plot disimpan ke training_history_optimized.png')

In [ ]:
# ── EVALUASI TEST SET ──────────────────────────────────────
print('=' * 60)
print('Evaluasi Test Set — Best Model (by Macro F1)')
print('=' * 60)

model.load_state_dict(torch.load(model_save_path, map_location=device))
model.eval()

correct_birads   = 0
correct_density  = 0
total            = 0
all_prob_birads  = []
all_prob_density = []
all_true_birads  = []
all_true_density = []
all_pred_birads  = []
all_pred_density = []

with torch.no_grad():
    for inputs_cc, inputs_mlo, labels_birads, labels_density in test_loader:
        inputs_cc      = inputs_cc.float().to(device)
        inputs_mlo     = inputs_mlo.float().to(device)
        labels_birads  = labels_birads.long().to(device)
        labels_density = labels_density.long().to(device)

        pred_birads, pred_density = model(inputs_cc, inputs_mlo)

        total            += labels_birads.size(0)
        correct_birads   += (pred_birads.argmax(dim=1)  == labels_birads).sum().item()
        correct_density  += (pred_density.argmax(dim=1) == labels_density).sum().item()

        all_prob_birads.append(F.softmax(pred_birads,   dim=1).cpu().numpy())
        all_prob_density.append(F.softmax(pred_density, dim=1).cpu().numpy())
        all_true_birads.append(labels_birads.cpu().numpy())
        all_true_density.append(labels_density.cpu().numpy())
        all_pred_birads.append(pred_birads.argmax(dim=1).cpu().numpy())
        all_pred_density.append(pred_density.argmax(dim=1).cpu().numpy())

test_acc_birads  = 100 * correct_birads  / total
test_acc_density = 100 * correct_density / total

all_prob_birads  = np.concatenate(all_prob_birads,  axis=0)
all_prob_density = np.concatenate(all_prob_density, axis=0)
all_true_birads  = np.concatenate(all_true_birads,  axis=0)
all_true_density = np.concatenate(all_true_density, axis=0)
all_pred_birads  = np.concatenate(all_pred_birads,  axis=0)
all_pred_density = np.concatenate(all_pred_density, axis=0)

auc_birads  = metrics.roc_auc_score(
    label_binarize(all_true_birads,  classes=[0,1,2,3,4]),
    all_prob_birads,  multi_class='ovr', average='macro'
)
auc_density = metrics.roc_auc_score(
    label_binarize(all_true_density, classes=[0,1,2,3]),
    all_prob_density, multi_class='ovr', average='macro'
)

# [BARU] Hitung Macro F1 dan Balanced Accuracy di test set
f1_birads_test   = metrics.f1_score(all_true_birads,  all_pred_birads,  average='macro', zero_division=0)
f1_density_test  = metrics.f1_score(all_true_density, all_pred_density, average='macro', zero_division=0)
bal_acc_birads   = metrics.balanced_accuracy_score(all_true_birads,  all_pred_birads)
bal_acc_density  = metrics.balanced_accuracy_score(all_true_density, all_pred_density)

print(f'\n  BI-RADS → Accuracy: {test_acc_birads:.2f}% | '
      f'Macro F1: {f1_birads_test:.4f} | '
      f'Balanced Acc: {bal_acc_birads:.4f} | '
      f'AUC: {auc_birads:.4f}')
print(f'  Density → Accuracy: {test_acc_density:.2f}% | '
      f'Macro F1: {f1_density_test:.4f} | '
      f'Balanced Acc: {bal_acc_density:.4f} | '
      f'AUC: {auc_density:.4f}')

print('\n── Classification Report BI-RADS ──')
print(metrics.classification_report(
    all_true_birads, all_pred_birads,
    target_names=['BI-RADS 1','BI-RADS 2','BI-RADS 3','BI-RADS 4','BI-RADS 5'],
    zero_division=0
))

print('── Classification Report Density ──')
print(metrics.classification_report(
    all_true_density, all_pred_density,
    target_names=['Density A','Density B','Density C','Density D'],
    zero_division=0
))

In [ ]:
# ── CONFUSION MATRIX ───────────────────────────────────────
# Penting untuk melihat MANA kelas yang masih salah diprediksi
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_birads  = metrics.confusion_matrix(all_true_birads,  all_pred_birads)
cm_density = metrics.confusion_matrix(all_true_density, all_pred_density)

disp_b = metrics.ConfusionMatrixDisplay(
    cm_birads,
    display_labels=['BI-RADS 1','BI-RADS 2','BI-RADS 3','BI-RADS 4','BI-RADS 5']
)
disp_d = metrics.ConfusionMatrixDisplay(
    cm_density,
    display_labels=['Density A','Density B','Density C','Density D']
)

disp_b.plot(ax=axes[0], colorbar=False, cmap='Blues')
disp_d.plot(ax=axes[1], colorbar=False, cmap='Greens')

axes[0].set_title('Confusion Matrix — BI-RADS')
axes[1].set_title('Confusion Matrix — Density')

plt.tight_layout()
plt.savefig('confusion_matrix_optimized.png', dpi=150)
plt.show()
print('Confusion matrix disimpan ke confusion_matrix_optimized.png')